# 05 Review outputs

Review the latest inventory, rule-classification, and text-extraction outputs together before any rename planning or execution.


In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

OUTPUTS_DIR = PROJECT_ROOT / 'data' / 'outputs'
EXPORT_REVIEW_SNAPSHOT = True


In [2]:
from src.reporting import detect_latest_outputs, load_optional_parquet, build_review_frame, review_summary
from src.inventory import ensure_inventory_schema

paths = detect_latest_outputs(OUTPUTS_DIR)
print(paths)

inv = load_optional_parquet(paths.inventory_path)
classified = load_optional_parquet(paths.classification_path)
text_df = load_optional_parquet(paths.text_path)

if inv is not None:
    inv = ensure_inventory_schema(inv)

review = build_review_frame(inv, classified, text_df)
print('Rows in review frame:', len(review))


ReviewPaths(inventory_path=WindowsPath('c:/00_Developement/sch-file-organizer/data/outputs/inventory_Random_Files_WORKING_COPY_20260308_115612.parquet'), classification_path=WindowsPath('c:/00_Developement/sch-file-organizer/data/outputs/rule_classification_20260308_130634.parquet'), text_path=WindowsPath('c:/00_Developement/sch-file-organizer/data/outputs/inventory_with_text_20260308_144155.parquet'))
Rows in review frame: 1806


In [3]:
summary = review_summary(review)
pd.DataFrame([summary])


,rows,duplicates,junk_candidates,manual_review,text_errors,long_paths
0,1806,260,0,1676,1,140


In [5]:
display(review[['relative_path', 'suffix', 'size_bytes', 'rule_status', 'text_status', 'path_length']].head(20))
display(review['rule_status'].value_counts(dropna=False).rename_axis('rule_status').reset_index(name='count'))
display(review['text_status'].value_counts(dropna=False).rename_axis('text_status').reset_index(name='count'))
display(review['suffix'].fillna('').value_counts(dropna=False).rename_axis('suffix').reset_index(name='count').head(20))


,relative_path,suffix,size_bytes,rule_status,text_status,path_length
0,01_selected\random_1_dedup_target\doclaynet_pd...,.pdf,273096,move_to_special_folder,ok,123
1,01_selected\random_1_dedup_target\doclaynet_pd...,.txt,5646,move_to_special_folder,ok,123
2,01_selected\random_1_dedup_target\doclaynet_pd...,.pdf,60106,move_to_special_folder,ok,123
3,01_selected\random_1_dedup_target\doclaynet_pd...,.txt,2232,move_to_special_folder,ok,123
4,01_selected\random_1_dedup_target\doclaynet_pd...,.pdf,183832,move_to_special_folder,ok,124
5,01_selected\random_1_dedup_target\doclaynet_pd...,.txt,3197,move_to_special_folder,ok,124
6,01_selected\random_1_dedup_target\doclaynet_pd...,.pdf,125583,move_to_special_folder,ok,133
7,01_selected\random_1_dedup_target\doclaynet_pd...,.txt,8218,move_to_special_folder,ok,133
8,01_selected\random_1_dedup_target\doclaynet_pd...,.pdf,38215,move_to_special_folder,ok,119
9,01_selected\random_1_dedup_target\doclaynet_pd...,.txt,2467,move_to_special_folder,ok,119


,rule_status,count
0,review,1676
1,move_to_special_folder,130


,text_status,count
0,ok,825
1,unsupported,666
2,empty,314
3,error,1


,suffix,count
0,.pdf,682
1,.txt,353
2,.xls,213
3,.doc,165
4,.stp,156
5,.html,86
6,.png,75
7,.ppt,50
8,.csv,17
9,.rtf,5


In [6]:
junk = review[review['rule_status'] == 'archive_or_delete_candidate'].copy()
duplicates = review[review.get('is_duplicate_hash', False).fillna(False)].copy()
manual_review = review[review['needs_manual_review']].copy()
text_errors = review[review['has_text_error']].copy()
long_paths = review[review['long_path_warning']].copy()
compliant = review[review['rule_status'] == 'compliant_keep_review_path'].copy()

display(junk[['relative_path', 'rule_reason', 'proposed_relative_target']].head(20))
display(duplicates[['relative_path', 'hash', 'duplicate_group_size', 'rule_status']].head(20))
display(text_errors[['relative_path', 'suffix', 'text_source', 'text_error']].head(20))
display(long_paths[['relative_path', 'path_length', 'filename_length', 'rule_status']].sort_values('path_length', ascending=False).head(20))
display(compliant[['relative_path', 'parsed_phase', 'parsed_doc_type', 'proposed_relative_target']].head(20))
display(manual_review[['relative_path', 'suffix', 'rule_reason', 'text_status', 'text_preview']].head(30))


,relative_path,rule_reason,proposed_relative_target


,relative_path,hash,duplicate_group_size,rule_status
0,01_selected\random_1_dedup_target\doclaynet_pd...,5277953e8a59ada7695221174c70ab83,2,move_to_special_folder
1,01_selected\random_1_dedup_target\doclaynet_pd...,26bd341bc4c020311ae9b282384e4dc9,2,move_to_special_folder
2,01_selected\random_1_dedup_target\doclaynet_pd...,cd68043be50b2e66ddc147e1dcb65d1c,2,move_to_special_folder
3,01_selected\random_1_dedup_target\doclaynet_pd...,29632aa2b0937e9f051d357afc0ea63d,2,move_to_special_folder
4,01_selected\random_1_dedup_target\doclaynet_pd...,edcd22f784f2948ece88d5b2b2c8b142,2,move_to_special_folder
5,01_selected\random_1_dedup_target\doclaynet_pd...,8471f4848321d8459bf941ffde8caf2a,2,move_to_special_folder
6,01_selected\random_1_dedup_target\doclaynet_pd...,5fe0c32fd8b3292f1d686a91b9020ba4,2,move_to_special_folder
7,01_selected\random_1_dedup_target\doclaynet_pd...,c1717e891ef13f25302fb7b97d10fba6,2,move_to_special_folder
8,01_selected\random_1_dedup_target\doclaynet_pd...,eb52d09a188210669ced33b59b23a196,2,move_to_special_folder
9,01_selected\random_1_dedup_target\doclaynet_pd...,15134bea88392a764225998f5ec33847,2,move_to_special_folder


,relative_path,suffix,text_source,text_error
808,01_selected\govdocs1\govdocs1_0068_002400.pdf,.pdf,pdf,PdfReadError: Invalid Elementary Object starti...


,relative_path,path_length,filename_length,rule_status
1666,01_selected\very_very_very_very_very_very_very...,297,35,review
1667,01_selected\very_very_very_very_very_very_very...,297,35,review
1668,01_selected\very_very_very_very_very_very_very...,297,35,review
1669,01_selected\very_very_very_very_very_very_very...,297,35,review
1670,01_selected\very_very_very_very_very_very_very...,297,35,review
1671,01_selected\very_very_very_very_very_very_very...,297,35,review
1672,01_selected\very_very_very_very_very_very_very...,297,35,review
1673,01_selected\very_very_very_very_very_very_very...,297,35,review
1674,01_selected\very_very_very_very_very_very_very...,297,35,review
1675,01_selected\very_very_very_very_very_very_very...,297,35,review


,relative_path,parsed_phase,parsed_doc_type,proposed_relative_target


,relative_path,suffix,rule_reason,text_status,text_preview
130,01_selected\New Text Document.ogb,.ogb,needs classification or rename mapping,unsupported,
131,01_selected\doclaynet_pdf\doclaynet_pdf_0001_r...,.pdf,needs classification or rename mapping,ok,InnoDB Transaction Model • REPEATABLE READ Thi...
132,01_selected\doclaynet_pdf\doclaynet_pdf_0001_r...,.txt,needs classification or rename mapping,ok,InnoDB Transaction Model • REPEATABLE READ •Fo...
133,01_selected\doclaynet_pdf\doclaynet_pdf_0002_p...,.pdf,needs classification or rename mapping,ok,10-12
134,01_selected\doclaynet_pdf\doclaynet_pdf_0002_p...,.txt,needs classification or rename mapping,ok,10-12
135,01_selected\doclaynet_pdf\doclaynet_pdf_0003_1...,.pdf,needs classification or rename mapping,ok,"In the incoherent regime, the adiabatic elimin..."
136,01_selected\doclaynet_pdf\doclaynet_pdf_0003_1...,.txt,needs classification or rename mapping,ok,"In the incoherent regime, the adiabatic elimin..."
137,01_selected\doclaynet_pdf\doclaynet_pdf_0004_N...,.pdf,needs classification or rename mapping,ok,152 Non-Corporate CDOs and Other Derivative Tr...
138,01_selected\doclaynet_pdf\doclaynet_pdf_0004_N...,.txt,needs classification or rename mapping,ok,Non-Corporate CDOs and Other Derivative Transa...
139,01_selected\doclaynet_pdf\doclaynet_pdf_0005_N...,.pdf,needs classification or rename mapping,ok,-42- UTILIZATION: Fiscal Year 2010 First Secon...


In [7]:
review_candidates = review[review['needs_manual_review'] | review['has_text_error'] | review['long_path_warning']].copy()
review_candidates = review_candidates.sort_values(['needs_manual_review', 'has_text_error', 'path_length', 'relative_path'], ascending=[False, False, False, True])
display(review_candidates[['relative_path', 'suffix', 'rule_status', 'rule_reason', 'text_status', 'path_length', 'text_preview']].head(50))


,relative_path,suffix,rule_status,rule_reason,text_status,path_length,text_preview
808,01_selected\govdocs1\govdocs1_0068_002400.pdf,.pdf,review,needs classification or rename mapping,error,93,
1666,01_selected\very_very_very_very_very_very_very...,.pdf,review,path length warning,empty,297,
1667,01_selected\very_very_very_very_very_very_very...,.txt,review,path length warning,ok,297,KING'S CONFECTIONERY S/B 273500-0 (GKJ) LOT NO...
1668,01_selected\very_very_very_very_very_very_very...,.pdf,review,path length warning,empty,297,
1669,01_selected\very_very_very_very_very_very_very...,.txt,review,path length warning,ok,297,"AEON CO. (M) BHD (126926-H) 3RD FLR, AEON TAMA..."
1670,01_selected\very_very_very_very_very_very_very...,.pdf,review,path length warning,empty,297,
1671,01_selected\very_very_very_very_very_very_very...,.txt,review,path length warning,ok,297,HON HWA HARDWARE TRADING Company Reg. No. : 00...
1672,01_selected\very_very_very_very_very_very_very...,.pdf,review,path length warning,empty,297,
1673,01_selected\very_very_very_very_very_very_very...,.txt,review,path length warning,ok,297,TOKYO KITCHEN (CITTA MALL) TOKYO KITCHEN SDN B...
1674,01_selected\very_very_very_very_very_very_very...,.pdf,review,path length warning,empty,297,


In [8]:
if EXPORT_REVIEW_SNAPSHOT:
    OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
    review_path = OUTPUTS_DIR / 'review_snapshot_latest.parquet'
    review_csv = OUTPUTS_DIR / 'review_snapshot_latest.csv'
    review.to_parquet(review_path, index=False)
    review.to_csv(review_csv, index=False, encoding='utf-8-sig')
    print('Saved:', review_path)
    print('Saved:', review_csv)


Saved: c:\00_Developement\sch-file-organizer\data\outputs\review_snapshot_latest.parquet
Saved: c:\00_Developement\sch-file-organizer\data\outputs\review_snapshot_latest.csv
